# CaptionScore: fine-tune DistilBERT on Colab (Phase 4b)

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

You need two files from the project (made by `python -m src.export_colab`):
1. `ml/data/colab/captionscore_finetune.parquet`: train subsample + validation (no test/stress)
2. `ml/src/finetune_transformer.py`: the training script (the single source of truth; this
   notebook just runs it)

At the end you download **`distilbert_results.zip`** and unzip it into
`ml/data/colab/results/` in the project, then run `python -m src.import_transformer_preds`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers==5.24.1" "accelerate>=1.1.0"

## 1. Upload the two files
Use the file picker below. Alternatively, put them in Google Drive and copy them over.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select captionscore_finetune.parquet AND finetune_transformer.py
import os
assert os.path.exists("captionscore_finetune.parquet") and os.path.exists("finetune_transformer.py"), "upload both files"

In [ ]:
import pandas as pd
df = pd.read_parquet("captionscore_finetune.parquet")
print(df["split"].value_counts(), "
positive rate:", df["viral"].mean().round(4))
assert set(df["split"]) <= {"train", "val"}

## 2. Sanity check: shuffled labels (optional, a few minutes)
This trains on 30k rows with **shuffled** labels. Its validation PR-AUC should come out near
the 0.10 chance level. If it's much higher, something is leaking.

In [ ]:
!python finetune_transformer.py --data captionscore_finetune.parquet --out out --shuffle-labels --max-train-rows 30000 --epochs 1

## 3. Main run
DistilBERT, max_length 64, fp16, 2 epochs, lr 3e-5, batch 128. The best checkpoint is chosen
on the **inner hold-out** (last 3 training months), never on validation.
Rough estimate on a T4: 10-30 min (not measured).

In [ ]:
!python finetune_transformer.py --data captionscore_finetune.parquet --out out --epochs 2

## 4. Download the results
The zip holds the predictions and run info (small). The fine-tuned model is zipped separately (~250 MB); only download it if DistilBERT wins on validation.

In [ ]:
import shutil, glob, os
os.makedirs("results", exist_ok=True)
for f in glob.glob("out/val_predictions_distilbert*.parquet") + glob.glob("out/run_info*.json"):
    shutil.copy(f, "results/")
print(os.listdir("results"))
shutil.make_archive("distilbert_results", "zip", "results")
files.download("distilbert_results.zip")

In [ ]:
# OPTIONAL: the fine-tuned model (~250 MB). Only needed if DistilBERT is chosen as the final model.
# shutil.make_archive("distilbert_model", "zip", "out/model")
# files.download("distilbert_model.zip")